# LASSO Regression

**Project question:** Can L1 shrinkage produce a sparse predictive model without turning selection into causal evidence?

By the end of this notebook, you should be able to:

- tune LASSO alpha inside a standardized pipeline
- identify coefficients set exactly to zero
- evaluate predictive performance and selection instability cautiously

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [ ]:

from lite_setup import ensure_packages
await ensure_packages()

LASSO minimizes $\sum_i (y_i-\hat y_i)^2 + \alpha\sum_j |\beta_j|$. Its L1 geometry can set slopes exactly to zero, but which member of a correlated group survives can be sample dependent.

In [ ]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_squared_error

In [ ]:
df = pd.read_csv(DATA / 'simulated_correlated_predictors.csv')
X = df.drop(columns=['id', 'weekly_sales'])
y = df['weekly_sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=4031, test_size=0.30)
cv = KFold(n_splits=5, shuffle=True, random_state=4031)
pipe = make_pipeline(StandardScaler(), Lasso(max_iter=50000))
search = GridSearchCV(
    pipe, {'lasso__alpha': np.logspace(-3, 1, 30)},
    scoring='neg_root_mean_squared_error', cv=cv,
)
search.fit(X_train, y_train)
search.best_params_, -search.best_score_

In [ ]:
def rmse(actual, predicted):
    return float(np.sqrt(mean_squared_error(actual, predicted)))

ols = LinearRegression().fit(X_train, y_train)
pd.DataFrame([
    {'model': 'OLS', 'test_rmse': rmse(y_test, ols.predict(X_test))},
    {'model': 'tuned LASSO', 'test_rmse': rmse(y_test, search.predict(X_test))},
])

In [ ]:
lasso = search.best_estimator_.named_steps['lasso']
coef = pd.Series(lasso.coef_, index=X.columns, name='coefficient_per_1_sd_increase')
selection_table = pd.DataFrame({
    'coefficient_per_1_sd_increase': coef,
    'selected': coef.abs() > 1e-8,
}).sort_values('coefficient_per_1_sd_increase', key=np.abs, ascending=False)
selection_table

**Interpretation and caution:** With correlated predictors, LASSO may keep one member of a group and drop another across nearby samples. `selected=True` describes this predictive fit; it is not a hypothesis test, causal discovery, or guarantee of future selection stability.